In [1]:
from pathlib import Path
import sys
import hashlib
import json
import pandas as pd

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
SRC_DIR = PROJECT_ROOT / "src"

# A disk fingerprint must not describe stale Python classes in a running kernel.
if any(name == "toxic_comments" or name.startswith("toxic_comments.") for name in sys.modules):
    raise RuntimeError("Restart the kernel, then run from the first cell (skip the optional download).")
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))


def source_fingerprint():
    digest = hashlib.sha256()
    for path in sorted((SRC_DIR / "toxic_comments").rglob("*.py")):
        digest.update(str(path.relative_to(SRC_DIR)).encode())
        digest.update(path.read_bytes())
    return digest.hexdigest()


SESSION_SOURCE_FINGERPRINT = source_fingerprint()


def experiment_fingerprint(train, validation, text_column, **settings):
    if source_fingerprint() != SESSION_SOURCE_FINGERPRINT:
        raise RuntimeError("Source changed: restart the kernel and rerun training and validation.")
    digest = hashlib.sha256(SESSION_SOURCE_FINGERPRINT.encode())
    digest.update(json.dumps(settings, sort_keys=True, default=str).encode())
    columns = (["id"] if "id" in train and "id" in validation else []) + [text_column] + LABEL_COLUMNS
    digest.update(json.dumps(columns).encode())
    for name, frame in [("train", train), ("validation", validation)]:
        digest.update(f"{name}:{len(frame)}".encode())
        digest.update(pd.util.hash_pandas_object(frame[columns], index=False).values.tobytes())
    return digest.hexdigest()


def completed_configurations_for(rows, fingerprint):
    return {
        (row["model_name"], row["pooling"])
        for row in rows
        if row.get("experiment_fingerprint") == fingerprint
        and pd.isna(row.get("error"))
        and pd.notna(row.get("macro_roc_auc"))
    }

In [2]:
from toxic_comments.config import PROCESSED_DATA_DIR
df = pd.read_csv(PROCESSED_DATA_DIR / "train_clean.csv")

In [3]:
df[df["is_empty_heavy"] != 0]

,id,comment_text,toxic,severe_toxic,obscene,threat,insult,identity_hate,comment_light,comment_heavy,...,punct_ratio,n_newlines,n_urls,n_ips,n_you,n_masked_words,has_shouting,is_empty_light,is_empty_heavy,is_non_latin
2405,067638a445ccd93b,"Here, here and here.",0,0,0,0,0,0,"Here, here and here.",NaN,...,0.100000,0,0,0,0,0,0,0,1,1
3987,0aa6f3529219b37e,From here\n\nFrom here 160.80.2.8,0,0,0,0,0,0,From here From here,NaN,...,0.096774,2,0,1,0,0,0,0,1,1
4479,0bed2196c873636d,1993\n\n1994\n\n1995\n\n1996\n\n1997\n\n1998\n...,0,0,0,0,0,0,1993 1994 1995 1996 1997 1998 1999 2000 2001 2...,NaN,...,0.000000,20,0,0,0,0,0,0,1,1
6296,10d0c3263b52a057,193.61.111.53 15:00,0,0,0,0,0,0,15:00,NaN,...,0.200000,0,0,1,0,0,0,0,1,1
8841,1776b3bcff81788b,What is I 78.146.102.144,0,0,0,0,0,0,What is I,NaN,...,0.125000,0,0,1,0,0,0,0,1,1
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
148773,5352b339650c4138,she did 76.122.79.82,0,0,0,0,0,0,she did,NaN,...,0.150000,0,0,1,0,0,0,0,1,1
148947,5631d491869e9f64,"- 00:49, 14 Feb 2005 (UTC)",0,0,0,0,0,0,-,NaN,...,0.185185,0,0,0,0,0,0,0,1,1
151286,7c1a57063b5c14d5,10 - 2010 04 08 to 2010 05 12,0,0,0,0,0,0,10 - 2010 04 08 to 2010 05 12,NaN,...,0.034483,0,0,0,0,0,0,0,1,1
153221,9af7112c4d554edb,which is OVER 9000 OVER 9000 OVER 9000 OVER 90...,0,0,0,0,0,0,which is OVER 9000 OVER 9000 OVER 9000 OVER 90...,NaN,...,0.000000,0,0,0,0,0,1,0,1,1


### create k-folds

In [4]:
from toxic_comments.splits import save_kfold_datasets

files = save_kfold_datasets(df)
files

[PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_1/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_1/test.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_2/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_2/test.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_3/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_3/test.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_4/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_4/test.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_5/train.csv'),
 PosixPath('/home/jkl0909/minhhuyen/toxic-comments/data/k_fold/fold_5/test.csv')]

In [5]:
from toxic_comments.config import K_FOLD_DATA_DIR, LABEL_COLUMNS

rows = []
for fold_index in range(1, 6):
    fold_dir = K_FOLD_DATA_DIR / f"fold_{fold_index}"
    for split_name in ["train", "test"]:
        split_path = fold_dir / f"{split_name}.csv"
        split_df = pd.read_csv(split_path)

        row = {
            "fold": fold_index,
            "split": split_name,
            "rows": len(split_df),
        }
        row.update({label: int(split_df[label].sum()) for label in LABEL_COLUMNS})
        rows.append(row)

fold_label_summary = pd.DataFrame(rows)
fold_label_summary


,fold,split,rows,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,1,train,127580,12230,1273,6756,381,6300,1125
1,1,test,31896,3060,321,1690,97,1574,280
2,2,train,127581,12231,1275,6756,383,6299,1123
3,2,test,31895,3059,319,1690,95,1575,282
4,3,train,127581,12233,1275,6755,384,6299,1125
5,3,test,31895,3057,319,1691,94,1575,280
6,4,train,127581,12233,1276,6759,382,6297,1125
7,4,test,31895,3057,318,1687,96,1577,280
8,5,train,127581,12233,1277,6758,382,6301,1122
9,5,test,31895,3057,317,1688,96,1573,283


### divide into train-validation-test

In [6]:
from toxic_comments.splits import train_validation_test_split
from toxic_comments.config import LABEL_COLUMNS

train_df, validation_df, test_df = train_validation_test_split(
    df,
    validation_size=0.1,
    test_size=0.1,
    random_state=42,
    stratified=True,
)

rows = []
for split_name, split_df in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    row = {
        "split": split_name,
        "rows": len(split_df),
    }
    row.update({label: int(split_df[label].sum()) for label in LABEL_COLUMNS})
    rows.append(row)

holdout_label_summary = pd.DataFrame(rows)
holdout_label_summary


,split,rows,toxic,severe_toxic,obscene,threat,insult,identity_hate
0,train,127580,12232,1275,6757,384,6299,1123
1,validation,15948,1529,162,845,44,786,143
2,test,15948,1529,157,844,50,789,139


### Test training with tf-idf

In [8]:
from toxic_comments.config import MODELS_DIR
from toxic_comments.train import save_model, train_model

model_name = "tfidf_logistic_regression"
text_column = "comment_heavy"
baseline_fingerprint = experiment_fingerprint(
    train_df, validation_df, text_column, max_features=20_000,
)

estimator = train_model(
    data=train_df,
    model_name=model_name,
    text_column=text_column,
    max_features=20_000,
)

model_path = save_model(
    estimator,
    MODELS_DIR / "notebook_test" / f"{model_name}.joblib",
)

print(f"Saved model: {model_path}")


Saved model: /home/jkl0909/minhhuyen/toxic-comments/models/notebook_test/tfidf_logistic_regression.joblib


In [9]:
from toxic_comments.config import RESULTS_DIR
from toxic_comments.train import evaluate_model

if baseline_fingerprint != experiment_fingerprint(
    train_df, validation_df, text_column, max_features=20_000,
):
    raise RuntimeError("Rerun TF-IDF training on the current split first.")

validation_metrics = {
    "experiment_fingerprint": baseline_fingerprint,
    "text_column": text_column,
    "model_name": model_name,
    "split": "validation",
    "rows": len(validation_df),
}
validation_metrics.update(
    evaluate_model(estimator, validation_df, text_column=text_column)
)

validation_metrics_df = pd.DataFrame([validation_metrics])
metrics_path = RESULTS_DIR / "notebook_test" / f"{model_name}_validation_metrics.csv"
metrics_path.parent.mkdir(parents=True, exist_ok=True)
validation_metrics_df.to_csv(metrics_path, index=False)

print(f"Saved validation metrics: {metrics_path}")
validation_metrics_df


Saved validation metrics: /home/jkl0909/minhhuyen/toxic-comments/results/notebook_test/tfidf_logistic_regression_validation_metrics.csv


,experiment_fingerprint,text_column,model_name,split,rows,subset_accuracy,hamming_loss,micro_precision,micro_recall,micro_f1,macro_f1,micro_roc_auc,macro_roc_auc
0,a254930447626ecf308f71b9e4dcbd3aecdc3d23da9efb...,comment_heavy,tfidf_logistic_regression,validation,15948,0.878731,0.029126,0.568449,0.854374,0.682682,0.559379,0.982115,0.975407


### to download word2vec embeddings

In [5]:
import gensim.downloader as api
from toxic_comments.config import EMBEDDINGS_DIR

path = EMBEDDINGS_DIR / "word2vec" / "GoogleNews-vectors-negative300.txt"
path.parent.mkdir(parents=True, exist_ok=True)

vectors = api.load("word2vec-google-news-300")
vectors.save_word2vec_format(str(path), binary=False)

KeyboardInterrupt: 

## 6 pooling × 5 embeddings vs TF-IDF

Reuse train_df / validation_df from the holdout split and the TF-IDF validation metrics above.
Each configuration = pretrained embeddings + pooling + StandardScaler + one-vs-rest LogisticRegression.
Restart the kernel after source edits. Progress uses a fingerprinted CSV per experiment; only matching successful configurations are skipped. Legacy `pooling_comparison.csv` is historical, not reusable. Fingerprints cover source, train/validation content and settings; increment `embedding_revision` when replacing embedding assets or changing dependency/runtime versions. Test data is not used for model selection.


In [10]:
from toxic_comments.config import RESULTS_DIR
from toxic_comments.train import train_model, evaluate_model

embedding_model_names = [
    "fasttext_logistic_regression",
    "glove_twitter_logistic_regression",
    "bpemb_logistic_regression",
    "word2vec_logistic_regression",
    "lexvec_logistic_regression",
]
poolings = [
    "mean",
    "max",
    "attention",
    "max_attention",
    "supervised_attention",
    "max_supervised_attention",
]
embedding_text_column = "comment_heavy"
embedding_revision = "pretrained-v1"  # Bump after replacing vector assets or runtime dependencies.
max_embedding_vectors = None  # Full word vocabularies; BPEmb always uses all subwords.
embedding_results_dir = RESULTS_DIR / "notebook_test" / "embeddings_max_attention"
embedding_results_dir.mkdir(parents=True, exist_ok=True)

for split_frame in (train_df, validation_df):
    if embedding_text_column not in split_frame:
        raise ValueError(f"Missing column: {embedding_text_column}")

In [11]:
import gc
from time import perf_counter

comparison_fingerprint = experiment_fingerprint(
    train_df, validation_df, embedding_text_column,
    max_embedding_vectors=max_embedding_vectors, embedding_revision=embedding_revision,
)
if baseline_fingerprint != experiment_fingerprint(
    train_df, validation_df, embedding_text_column, max_features=20_000,
) or validation_metrics_df["experiment_fingerprint"].iloc[0] != baseline_fingerprint:
    raise RuntimeError("Rerun TF-IDF training and validation on this split first.")
comparison_path = embedding_results_dir / f"pooling_comparison_{comparison_fingerprint}.csv"
comparison_rows = pd.read_csv(comparison_path).to_dict("records") if comparison_path.exists() else []
comparison_rows = [row for row in comparison_rows
                   if row.get("experiment_fingerprint") == comparison_fingerprint]
completed_configurations = completed_configurations_for(comparison_rows, comparison_fingerprint)

for embedding_model_name in embedding_model_names:
    for pooling_name in poolings:
        configuration = (embedding_model_name, pooling_name)
        if configuration in completed_configurations:
            print(f"Skipping completed configuration: {configuration}")
            continue

        print(f"Training and validating {embedding_model_name} with {pooling_name} pooling", flush=True)
        row = {
            "experiment_fingerprint": comparison_fingerprint,
            "model_name": embedding_model_name,
            "pooling": pooling_name,
            "text_column": embedding_text_column,
            "split": "validation",
            "train_rows": len(train_df),
            "rows": len(validation_df),
            "max_embedding_vectors": max_embedding_vectors,
            "fit_seconds": None,
            "eval_seconds": None,
            "error": None,
        }
        estimator = None
        try:
            started = perf_counter()
            estimator = train_model(
                data=train_df,
                model_name=embedding_model_name,
                text_column=embedding_text_column,
                embedding_pooling=pooling_name,
                max_embedding_vectors=max_embedding_vectors,
            )
            row["fit_seconds"] = perf_counter() - started
            started = perf_counter()
            row.update(evaluate_model(estimator, validation_df, text_column=embedding_text_column))
            row["eval_seconds"] = perf_counter() - started
        except Exception as exc:
            row["error"] = f"{type(exc).__name__}: {exc}"
            print(f"Failed {configuration}: {row['error']}", flush=True)
        finally:
            if estimator is not None:
                del estimator
            gc.collect()

        comparison_rows = [
            old for old in comparison_rows
            if (old.get("model_name"), old.get("pooling")) != configuration
        ]
        comparison_rows.append(row)
        pd.DataFrame(comparison_rows).to_csv(comparison_path, index=False)
        if row["error"] is None:
            completed_configurations.add(configuration)
        print(f"Saved progress to {comparison_path}", flush=True)

Skipping completed configuration: ('fasttext_logistic_regression', 'mean')
Skipping completed configuration: ('fasttext_logistic_regression', 'max')
Skipping completed configuration: ('fasttext_logistic_regression', 'attention')
Skipping completed configuration: ('fasttext_logistic_regression', 'max_attention')
Skipping completed configuration: ('fasttext_logistic_regression', 'supervised_attention')
Skipping completed configuration: ('fasttext_logistic_regression', 'max_supervised_attention')
Skipping completed configuration: ('glove_twitter_logistic_regression', 'mean')
Skipping completed configuration: ('glove_twitter_logistic_regression', 'max')
Skipping completed configuration: ('glove_twitter_logistic_regression', 'attention')
Training and validating glove_twitter_logistic_regression with max_attention pooling
Saved progress to /home/jkl0909/minhhuyen/toxic-comments/results/notebook_test/embeddings_max_attention/pooling_comparison_f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c955

In [12]:
if comparison_fingerprint != experiment_fingerprint(
    train_df, validation_df, embedding_text_column,
    max_embedding_vectors=max_embedding_vectors, embedding_revision=embedding_revision,
):
    raise RuntimeError("Comparison is stale: rerun training on the current split/settings.")
if validation_metrics_df["experiment_fingerprint"].iloc[0] != experiment_fingerprint(
    train_df, validation_df, embedding_text_column, max_features=20_000,
):
    raise RuntimeError("TF-IDF baseline is stale: rerun its training and validation.")
embedding_comparison_df = pd.DataFrame(comparison_rows)
if "macro_roc_auc" not in embedding_comparison_df:
    embedding_comparison_df["macro_roc_auc"] = float("nan")
embedding_comparison_df = embedding_comparison_df[
    embedding_comparison_df["model_name"].isin(embedding_model_names)
    & embedding_comparison_df["pooling"].isin(poolings)
]
failed = embedding_comparison_df[embedding_comparison_df["error"].notna()]
if not failed.empty:
    print("Failed configurations:")
    display(failed[["model_name", "pooling", "error"]])

successful = embedding_comparison_df[
    embedding_comparison_df["error"].isna() & embedding_comparison_df["macro_roc_auc"].notna()
]
tfidf_score = float(validation_metrics_df["macro_roc_auc"].iloc[0])

# 5 embeddings x 6 poolings, plus TF-IDF as a reference row.
auc_table = successful.pivot(index="model_name", columns="pooling", values="macro_roc_auc")
auc_table = auc_table.reindex(index=embedding_model_names, columns=poolings)
auc_table.loc["tfidf_logistic_regression"] = tfidf_score
auc_table["best"] = auc_table[poolings].max(axis=1)
print("macro_roc_auc on validation")
display(auc_table.style.format("{:.4f}", na_rep="-").highlight_max(axis=None, subset=poolings))

print("Difference vs TF-IDF (macro_roc_auc)")
display((auc_table.drop(index="tfidf_logistic_regression")[poolings] - tfidf_score)
        .style.format("{:+.4f}", na_rep="-"))

tfidf_baseline = validation_metrics_df.copy()
tfidf_baseline["pooling"] = "n/a"
tfidf_baseline["text_column"] = embedding_text_column
tfidf_baseline["error"] = None
comparison_with_tfidf = pd.concat(
    [tfidf_baseline, successful], ignore_index=True, sort=False
).sort_values("macro_roc_auc", ascending=False).reset_index(drop=True)
display(comparison_with_tfidf)

if successful.empty:
    print("No embedding configuration produced a macro_roc_auc score.")
else:
    best = successful.loc[successful["macro_roc_auc"].idxmax()]
    print(
        f"Best overall embedding: {best['model_name']} + {best['pooling']} | "
        f"macro_roc_auc={best['macro_roc_auc']:.6f} | "
        f"vs TF-IDF ({tfidf_score:.6f}): {best['macro_roc_auc'] - tfidf_score:+.6f}"
    )
    print("Best pooling per embedding:")
    for model_name, group in successful.groupby("model_name"):
        top = group.loc[group["macro_roc_auc"].idxmax()]
        print(f"  {model_name}: {top['pooling']} ({top['macro_roc_auc']:.6f})")
    print("Mean macro_roc_auc per pooling (across embeddings):")
    print(successful.groupby("pooling")["macro_roc_auc"].mean().reindex(poolings).to_string())

macro_roc_auc on validation


pooling,mean,max,attention,max_attention,supervised_attention,max_supervised_attention,best
model_name,,,,,,,
fasttext_logistic_regression,0.9724,0.9721,0.9725,0.9732,0.9422,0.9716,0.9732
glove_twitter_logistic_regression,0.9662,0.9564,0.9448,0.9634,0.9725,0.9798,0.9798
bpemb_logistic_regression,0.9537,0.9282,0.9330,0.9424,0.8864,0.9561,0.9561
word2vec_logistic_regression,0.9667,0.9539,0.9688,0.9687,0.9555,0.9715,0.9715
lexvec_logistic_regression,0.9658,0.9414,0.9580,0.9584,0.9563,0.9671,0.9671
tfidf_logistic_regression,0.9754,0.9754,0.9754,0.9754,0.9754,0.9754,0.9754


Difference vs TF-IDF (macro_roc_auc)


pooling,mean,max,attention,max_attention,supervised_attention,max_supervised_attention
model_name,,,,,,
fasttext_logistic_regression,-0.0030,-0.0033,-0.0029,-0.0022,-0.0332,-0.0038
glove_twitter_logistic_regression,-0.0092,-0.0190,-0.0307,-0.0120,-0.0029,+0.0044
bpemb_logistic_regression,-0.0217,-0.0472,-0.0424,-0.0330,-0.0890,-0.0193
word2vec_logistic_regression,-0.0087,-0.0215,-0.0066,-0.0067,-0.0199,-0.0039
lexvec_logistic_regression,-0.0096,-0.0340,-0.0174,-0.0171,-0.0191,-0.0083


,experiment_fingerprint,text_column,model_name,split,rows,subset_accuracy,hamming_loss,micro_precision,micro_recall,micro_f1,macro_f1,micro_roc_auc,macro_roc_auc,pooling,error,train_rows,max_embedding_vectors,fit_seconds,eval_seconds
0,f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c...,comment_heavy,glove_twitter_logistic_regression,validation,15948,0.837346,0.051846,0.408426,0.922770,0.566232,0.443613,0.982189,0.979785,max_supervised_attention,NaN,127580.0,NaN,389.548794,25.713621
1,a254930447626ecf308f71b9e4dcbd3aecdc3d23da9efb...,comment_heavy,tfidf_logistic_regression,validation,15948,0.878731,0.029126,0.568449,0.854374,0.682682,0.559379,0.982115,0.975407,n/a,None,NaN,NaN,NaN,NaN
2,f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c...,comment_heavy,fasttext_logistic_regression,validation,15948,0.832205,0.048512,0.424517,0.907951,0.578536,0.452480,0.979675,0.973230,max_attention,NaN,127580.0,NaN,279.262158,2.552044
3,f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c...,comment_heavy,fasttext_logistic_regression,validation,15948,0.808189,0.065640,0.347390,0.899117,0.501152,0.396385,0.974994,0.972548,attention,NaN,127580.0,NaN,119.837701,2.823056
4,f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c...,comment_heavy,glove_twitter_logistic_regression,validation,15948,0.821859,0.068452,0.339084,0.913081,0.494521,0.414328,0.976206,0.972515,supervised_attention,NaN,127580.0,NaN,297.598843,23.122860
5,f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c...,comment_heavy,fasttext_logistic_regression,validation,15948,0.805493,0.066685,0.343231,0.895982,0.496330,0.392738,0.974449,0.972446,mean,NaN,127580.0,NaN,104.280930,1.813153
6,f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c...,comment_heavy,fasttext_logistic_regression,validation,15948,0.791886,0.068483,0.337601,0.901681,0.491266,0.389506,0.972688,0.972143,max,NaN,127580.0,NaN,88.607282,1.606331
7,f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c...,comment_heavy,fasttext_logistic_regression,validation,15948,0.830574,0.051177,0.411412,0.918495,0.568280,0.440567,0.980775,0.971563,max_supervised_attention,NaN,127580.0,NaN,661.214926,40.331624
8,f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c...,comment_heavy,word2vec_logistic_regression,validation,15948,0.821733,0.058555,0.376095,0.905671,0.531483,0.410769,0.976986,0.971472,max_supervised_attention,NaN,127580.0,NaN,702.723718,51.681799
9,f40a5f76a0c6e31c92205c0fbda1623df136841d200f8c...,comment_heavy,word2vec_logistic_regression,validation,15948,0.801166,0.071702,0.325816,0.893417,0.477496,0.379883,0.970155,0.968815,attention,NaN,127580.0,NaN,198.799455,2.057579


Best overall embedding: glove_twitter_logistic_regression + max_supervised_attention | macro_roc_auc=0.979785 | vs TF-IDF (0.975407): +0.004378
Best pooling per embedding:
  bpemb_logistic_regression: max_supervised_attention (0.956123)
  fasttext_logistic_regression: max_attention (0.973230)
  glove_twitter_logistic_regression: max_supervised_attention (0.979785)
  lexvec_logistic_regression: max_supervised_attention (0.967147)
  word2vec_logistic_regression: max_supervised_attention (0.971472)
Mean macro_roc_auc per pooling (across embeddings):
pooling
mean                        0.964973
max                         0.950407
attention                   0.955425
max_attention               0.961218
supervised_attention        0.942579
max_supervised_attention    0.969218
